In [ ]:
from google.colab import files
uploaded = files.upload()

Saving demo_essay_dataset.csv to demo_essay_dataset.csv


In [ ]:
import pandas as pd

# Load Dataset
df = pd.read_csv("demo_essay_dataset.csv")

# Check Dataset
print(df.head())

# Prepare Data
essays = df['text'].tolist()
labels = df['label'].tolist()


                                                text  label
0  This essay discusses the role of education in ...      0
1  AI-generated essays often use formal language ...      1
2  Students often include personal experiences an...      0
3  The advancements in machine learning have enab...      1
4  Writing essays helps students to develop criti...      0


In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification, AdamW
import torch
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split

# Define the custom dataset class
class EssayDataset(Dataset):
    def __init__(self, essays, labels, tokenizer, max_length):
        self.essays = essays
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.essays)

    def __getitem__(self, idx):
        essay = self.essays[idx]
        label = self.labels[idx]

        encoding = self.tokenizer(
            essay,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label": torch.tensor(label, dtype=torch.long)
        }

# Initialize the tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
max_length = 512


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [ ]:
# Example essays and labels
essays = ["This is a human-written essay.", "This essay was generated by AI."]
labels = [0, 1]  # 0: human-written, 1: AI-generated

# Create an instance of EssayDataset
dataset = EssayDataset(essays, labels, tokenizer, max_length)

# View tokenized data for each example
for i in range(len(dataset)):
    tokenized_data = dataset[i]  # Retrieve the tokenized data
    print(f"Example {i + 1}:")
    print(f"Input IDs: {tokenized_data['input_ids']}")
    print(f"Attention Mask: {tokenized_data['attention_mask']}")
    print(f"Label: {tokenized_data['label']}")
    print("-" * 50)


Example 1:
Input IDs: tensor([ 101, 2023, 2003, 1037, 2529, 1011, 2517, 9491, 1012,  102,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0

In [ ]:
# Prepare data
essays = df['text'].tolist()
labels = df['label'].tolist()

# Split the dataset into training and validation sets
train_texts, val_texts, train_labels, val_labels = train_test_split(essays, labels, test_size=0.2)

# Create dataset objects
train_dataset = EssayDataset(train_texts, train_labels, tokenizer, max_length)
val_dataset = EssayDataset(val_texts, val_labels, tokenizer, max_length)

# Create DataLoader objects
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)


In [ ]:
class EssayClassifier:
    def __init__(self):
        self.model_name = "bert-base-uncased"
        self.model = BertForSequenceClassification.from_pretrained(self.model_name, num_labels=2)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)

    def train(self, train_loader, val_loader, epochs=4, lr=1e-5):
        optimizer = AdamW(self.model.parameters(), lr=lr)

        for epoch in range(epochs):
            print(f"Epoch {epoch + 1}/{epochs}")
            self.model.train()
            train_loss = 0

            for batch in train_loader:
                optimizer.zero_grad()
                input_ids = batch["input_ids"].to(self.device)
                attention_mask = batch["attention_mask"].to(self.device)
                labels = batch["label"].to(self.device)

                outputs = self.model(input_ids, attention_mask=attention_mask, labels=labels)
                loss = outputs.loss
                loss.backward()
                optimizer.step()

                train_loss += loss.item()

            avg_train_loss = train_loss / len(train_loader)
            print(f"Training Loss: {avg_train_loss}")

            # Validation
            self.model.eval()
            val_loss = 0
            correct = 0

            with torch.no_grad():
                for batch in val_loader:
                    input_ids = batch["input_ids"].to(self.device)
                    attention_mask = batch["attention_mask"].to(self.device)
                    labels = batch["label"].to(self.device)

                    outputs = self.model(input_ids, attention_mask=attention_mask, labels=labels)
                    val_loss += outputs.loss.item()

                    logits = outputs.logits
                    predictions = torch.argmax(logits, dim=1)
                    correct += (predictions == labels).sum().item()

            avg_val_loss = val_loss / len(val_loader)
            val_accuracy = correct / len(val_loader.dataset)
            print(f"Validation Loss: {avg_val_loss}, Accuracy: {val_accuracy}")

    def predict(self, essay):
        self.model.eval()
        encoding = tokenizer(essay, truncation=True, max_length=max_length, padding="max_length", return_tensors="pt")
        input_ids = encoding["input_ids"].to(self.device)
        attention_mask = encoding["attention_mask"].to(self.device)

        with torch.no_grad():
            outputs = self.model(input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            prediction = torch.argmax(logits, dim=1).item()

        return prediction


In [ ]:
# Create the classifier
classifier = EssayClassifier()

# Train the model
classifier.train(train_loader, val_loader, epochs=4)


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Epoch 1/4
Training Loss: 0.6627538204193115
Validation Loss: 0.7312407493591309, Accuracy: 0.5
Epoch 2/4
Training Loss: 0.6606650352478027
Validation Loss: 0.6874959468841553, Accuracy: 0.5
Epoch 3/4
Training Loss: 0.6280838251113892
Validation Loss: 0.6386464834213257, Accuracy: 0.5
Epoch 4/4
Training Loss: 0.4686926007270813
Validation Loss: 0.6053330898284912, Accuracy: 0.5


In [ ]:
# Test the model with a sample essay
test_essay = "This essay discusses the role of education in shaping the future of society."
prediction = classifier.predict(test_essay)

# Output the prediction (0: Student-Written, 1: AI-Generated)
print(f"Prediction: {prediction} (0: Student-Written, 1: AI-Generated)")


Prediction: 0 (0: Student-Written, 1: AI-Generated)
